# NGUYỄN THÀNH TRÍ - MSSV: 6451071079
## BÀI 4: PHÂN LỚP DỮ LIỆU IRIS BẰNG NEURAL NETWORK KẾT HỢP 10-FOLD CROSS VALIDATION
### (K-FOLD CROSS VALIDATION VỚI K = 10)

In [1]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import keras
from keras import layers, models

print("Đã nạp đầy đủ các thư viện cần thiết cho Bài 4.")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


I0000 00:00:1794280312.024980   66234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1794280313.323119   66234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Đã nạp đầy đủ các thư viện cần thiết cho Bài 4.


### Câu a: Đọc bộ dữ liệu Iris từ sklearn
Hướng dẫn: Sử dụng hàm `load_iris()` từ thư viện `sklearn.datasets` để đọc dữ liệu.

In [2]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

iris = load_iris()
print("Đã nạp thành công bộ dữ liệu Iris từ sklearn.")
print(f"Số lượng mẫu: {iris.data.shape[0]}, Số thuộc tính: {iris.data.shape[1]}")
print(f"Danh sách các lớp: {iris.target_names.tolist()}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Đã nạp thành công bộ dữ liệu Iris từ sklearn.
Số lượng mẫu: 150, Số thuộc tính: 4
Danh sách các lớp: ['setosa', 'versicolor', 'virginica']


### Câu b: Chuẩn hóa dữ liệu data về đoạn [0, 1]
Hướng dẫn: Gán X là data và Y là target. Chia mỗi giá trị của X cho giá trị lớn nhất trong cột tương ứng (dùng `X.max(axis=0)`).

In [3]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

X = iris.data.copy()
Y = iris.target.copy()

X_max = X.max(axis=0)
X = X / X_max

print("Đã chuẩn hóa ma trận thuộc tính X về đoạn [0, 1].")
print(f"Giá trị cực đại từng cột trước chuẩn hóa (X_max): {X_max}")
print(f"Giá trị nhỏ nhất sau chuẩn hóa: {X.min():.2f}")
print(f"Giá trị lớn nhất sau chuẩn hóa: {X.max():.2f}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Đã chuẩn hóa ma trận thuộc tính X về đoạn [0, 1].
Giá trị cực đại từng cột trước chuẩn hóa (X_max): [7.9 4.4 6.9 2.5]
Giá trị nhỏ nhất sau chuẩn hóa: 0.04
Giá trị lớn nhất sau chuẩn hóa: 1.00


### Câu c: Định nghĩa hàm create_model khởi tạo và biên dịch mạng Neural Network
Kiến trúc mô hình (tương tự Bài 1.h):
- Hidden layer 1: 10 nơ-ron, hàm kích hoạt ReLU, Fully connected, input_shape=(4,).
- Hidden layer 2: 20 nơ-ron, hàm kích hoạt ReLU, Fully connected.
- Output layer: 3 nơ-ron, hàm kích hoạt Softmax, Fully connected.
- Biên dịch với: optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'].

In [4]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def create_model():
    model = keras.Sequential([
        layers.Input(shape=(4,)),
        layers.Dense(10, activation='relu'),
        layers.Dense(20, activation='relu'),
        layers.Dense(3, activation='softmax')
    ], name="Iris_MLP_Classifier")
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Khởi tạo mô hình mẫu và in tóm tắt kiến trúc
sample_model = create_model()
sample_model.summary()

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


Model: "Iris_MLP_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

### Câu d: Huấn luyện mô hình bằng K-Fold Cross Validation (k = 10)
Hướng dẫn:
- Sử dụng `KFold(n_splits=10, shuffle=True, random_state=42)`.
- Trong quá trình huấn luyện, lưu và in ra kết quả Accuracy, Precision, Recall, F1-score của từng fold.
- Lưu lại các mô hình sau mỗi fold.

In [5]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models_list = []
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []

# Mảng lưu kết quả từng class qua 10 fold
per_class_recalls = {name: [] for name in iris.target_names}
per_class_precisions = {name: [] for name in iris.target_names}
per_class_f1s = {name: [] for name in iris.target_names}

print("Bắt đầu huấn luyện 10-Fold Cross Validation trên bộ dữ liệu Iris:")
print("-" * 80)
print(f"{'Fold':^6} | {'Accuracy':^12} | {'Precision':^12} | {'Recall':^12} | {'F1-Score':^12}")
print("-" * 80)

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train_f, X_val_f = X[train_idx], X[val_idx]
    y_train_f, y_val_f = Y[train_idx], Y[val_idx]

    model = create_model()
    model.fit(X_train_f, y_train_f, epochs=100, batch_size=8, verbose=0)

    y_val_prob = model.predict(X_val_f, verbose=0)
    y_val_pred = np.argmax(y_val_prob, axis=1)

    acc = accuracy_score(y_val_f, y_val_pred)
    prec = precision_score(y_val_f, y_val_pred, average='macro', zero_division=0)
    rec = recall_score(y_val_f, y_val_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val_f, y_val_pred, average='macro', zero_division=0)

    models_list.append(model)
    fold_accuracies.append(acc)
    fold_precisions.append(prec)
    fold_recalls.append(rec)
    fold_f1s.append(f1)

    # Thống kê chi tiết từng class
    report = classification_report(y_val_f, y_val_pred, target_names=iris.target_names, output_dict=True, zero_division=0)
    for c_name in iris.target_names:
        per_class_recalls[c_name].append(report[c_name]['recall'])
        per_class_precisions[c_name].append(report[c_name]['precision'])
        per_class_f1s[c_name].append(report[c_name]['f1-score'])

    print(f"{fold_idx:^6d} | {acc:^12.4f} | {prec:^12.4f} | {rec:^12.4f} | {f1:^12.4f}")

print("-" * 80)
print("Hoàn tất huấn luyện và đánh giá trên 10 folds.")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Bắt đầu huấn luyện 10-Fold Cross Validation trên bộ dữ liệu Iris:
--------------------------------------------------------------------------------
 Fold  |   Accuracy   |  Precision   |    Recall    |   F1-Score  
--------------------------------------------------------------------------------


  1    |    0.9333    |    0.9167    |    0.9444    |    0.9221   


  2    |    1.0000    |    1.0000    |    1.0000    |    1.0000   


  3    |    1.0000    |    1.0000    |    1.0000    |    1.0000   


  4    |    0.9333    |    0.9524    |    0.9333    |    0.9373   


  5    |    1.0000    |    1.0000    |    1.0000    |    1.0000   


  6    |    0.8667    |    0.8667    |    0.8889    |    0.8500   


  7    |    0.8667    |    0.9167    |    0.8333    |    0.8413   


  8    |    1.0000    |    1.0000    |    1.0000    |    1.0000   


  9    |    1.0000    |    1.0000    |    1.0000    |    1.0000   


  10   |    1.0000    |    1.0000    |    1.0000    |    1.0000   
--------------------------------------------------------------------------------
Hoàn tất huấn luyện và đánh giá trên 10 folds.


### Câu e: In ra Average Accuracy, Average Recall, Average Precision, Average F1-score của mô hình qua 10 fold
Hướng dẫn: Tính trung bình cộng của các độ đo đã lưu qua 10 fold.

In [6]:
# Nguyễn Thành Trí - MSSV: 6451071079
print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

avg_acc = np.mean(fold_accuracies)
avg_prec = np.mean(fold_precisions)
avg_rec = np.mean(fold_recalls)
avg_f1 = np.mean(fold_f1s)

print("Kết quả đánh giá trung bình qua 10-Fold Cross Validation:")
print(f"Average Accuracy:  {avg_acc:.4f} ({avg_acc * 100:.2f}%)")
print(f"Average Precision: {avg_prec:.4f} ({avg_prec * 100:.2f}%)")
print(f"Average Recall:    {avg_rec:.4f} ({avg_rec * 100:.2f}%)")
print(f"Average F1-score:  {avg_f1:.4f} ({avg_f1 * 100:.2f}%)")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Kết quả đánh giá trung bình qua 10-Fold Cross Validation:
Average Accuracy:  0.9600 (96.00%)
Average Precision: 0.9652 (96.52%)
Average Recall:    0.9600 (96.00%)
Average F1-score:  0.9551 (95.51%)
